# NB11 UTXOs and accounts — Chapter 11

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB11-1: UTXO state transition — remove from the set the outputs the inputs point to, and add the new outputs (for study)
import hashlib, json

def txid(tx: dict) -> str:
    return hashlib.sha256(json.dumps(tx, sort_keys=True).encode()).hexdigest()[:12]

def apply_utxo(S: dict, tx: dict):
    """S: {(txid, index): {"owner":..., "value":...}}.  tx: {"inputs":[(txid, index, signer)], "outputs":[(owner, value)]}
    The three APPLY steps of Buterin (2014) as they are. On success returns (new state, fee); on failure, an error string."""
    S = dict(S)
    in_sum = 0
    for (prev_id, idx, signer) in tx["inputs"]:
        key = (prev_id, idx)
        if key not in S:
            return f"error: input {key} is not an unspent output (already spent or nonexistent)"
        if S[key]["owner"] != signer:
            return f"error: the owner of input {key} is {S[key]['owner']} but the signer is {signer}"
        in_sum += S[key]["value"]
    out_sum = sum(v for _, v in tx["outputs"])
    if in_sum < out_sum:
        return f"error: input sum {in_sum} < output sum {out_sum}"
    for (prev_id, idx, _) in tx["inputs"]:
        del S[(prev_id, idx)]
    tid = txid(tx)
    for i, (owner, value) in enumerate(tx["outputs"]):
        S[(tid, i)] = {"owner": owner, "value": value}
    return S, in_sum - out_sum

def balance(S, who):                       # the "balance" is not in the ledger — it is added up each time
    return sum(o["value"] for o in S.values() if o["owner"] == who)

coinbase = {"inputs": [], "outputs": [("Alice", 10)]}          # 10 to Alice as a block reward
S = {(txid(coinbase), 0): {"owner": "Alice", "value": 10}}
print("initial UTXO set:", {k: v for k, v in S.items()})

tx1 = {"inputs": [(txid(coinbase), 0, "Alice")], "outputs": [("Bob", 3), ("Alice", 6.9)]}   # pay 3 + 6.9 change, 0.1 fee
S, fee = apply_utxo(S, tx1)
print("after tx1 — balance Alice:", balance(S, "Alice"), "Bob:", balance(S, "Bob"), "| fee:", round(fee, 2))
print("UTXO set size:", len(S), "(the one output of 10 is gone, and two outputs, 3 and 6.9, have appeared)")

tx2 = {"inputs": [(txid(coinbase), 0, "Alice")], "outputs": [("Carol", 10)]}                 # a double spend that tries to spend the same output again
print("tx2 (double spend):", apply_utxo(S, tx2))
tx3 = {"inputs": [(txid(tx1), 0, "Mallory")], "outputs": [("Mallory", 3)]}                 # someone else's output
print("tx3 (someone else's output):", apply_utxo(S, tx3))
tx4 = {"inputs": [(txid(tx1), 0, "Bob")], "outputs": [("Carol", 3)]}                       # Bob sends the 3 he received to Carol
S, fee = apply_utxo(S, tx4)
print("after tx4 — Bob:", balance(S, "Bob"), "Carol:", balance(S, "Carol"))

initial UTXO set: {('02c83c8a0105', 0): {'owner': 'Alice', 'value': 10}}
after tx1 — balance Alice: 6.9 Bob: 3 | fee: 0.1
UTXO set size: 2 (the one output of 10 is gone, and two outputs, 3 and 6.9, have appeared)
tx2 (double spend): error: input ('02c83c8a0105', 0) is not an unspent output (already spent or nonexistent)
tx3 (someone else's output): error: the owner of input ('99736848c857', 0) is Bob but the signer is Mallory
after tx4 — Bob: 0 Carol: 3


In [ ]:
# NB11-2: account state transition — subtract and add balances, but use the nonce to block "the same transaction twice" (for study)
def apply_account(S: dict, tx: dict):
    """S: {address: {"nonce": n, "balance": b}}.  tx: {"from", "to", "value", "nonce"}"""
    S = {k: dict(v) for k, v in S.items()}
    sender = S.setdefault(tx["from"], {"nonce": 0, "balance": 0})
    if tx["nonce"] != sender["nonce"]:
        return f"error: the next nonce of {tx['from']} is {sender['nonce']} but the transaction nonce is {tx['nonce']}"
    if sender["balance"] < tx["value"]:
        return f"error: balance {sender['balance']} < amount sent {tx['value']}"
    sender["nonce"] += 1
    sender["balance"] -= tx["value"]
    S.setdefault(tx["to"], {"nonce": 0, "balance": 0})["balance"] += tx["value"]
    return S

S = {"Alice": {"nonce": 0, "balance": 10}}
t1 = {"from": "Alice", "to": "Bob", "value": 3, "nonce": 0}
S = apply_account(S, t1)
print("after t1:", S)
print("applying the same t1 again (replay attack):", apply_account(S, t1))
t2 = {"from": "Alice", "to": "Carol", "value": 3, "nonce": 1}
S = apply_account(S, t2)
print("after t2:", S, "— there is no change output. The balance simply goes down")
t3 = {"from": "Alice", "to": "0x000…dead", "value": 4, "nonce": 2}      # to an address that does not exist (has no key) — a mistaken transfer
S = apply_account(S, t3)
print("after t3 (mistaken transfer):", S, "— the receiving account is created automatically and the balance is recorded. Only the key to that address can make a transaction that sends it back")

after t1: {'Alice': {'nonce': 1, 'balance': 7}, 'Bob': {'nonce': 0, 'balance': 3}}
applying the same t1 again (replay attack): error: the next nonce of Alice is 1 but the transaction nonce is 0
after t2: {'Alice': {'nonce': 2, 'balance': 4}, 'Bob': {'nonce': 0, 'balance': 3}, 'Carol': {'nonce': 0, 'balance': 3}} — there is no change output. The balance simply goes down
after t3 (mistaken transfer): {'Alice': {'nonce': 3, 'balance': 0}, 'Bob': {'nonce': 0, 'balance': 3}, 'Carol': {'nonce': 0, 'balance': 3}, '0x000…dead': {'nonce': 0, 'balance': 4}} — the receiving account is created automatically and the balance is recorded. Only the key to that address can make a transaction that sends it back
